In [ ]:
import torch

# =================================================================
# 1. 基础属性与形状变换 (Shapes & View)
# =================================================================
x = torch.arange(12)         # 生成 0 到 11 的向量
print(x.shape)               # 形状：torch.Size([12])
print(x.numel())             # 元素总数：12 (number of elements)

# reshape 的本质：改变对内存的“观察方式”，不改变底层数据存储
x = x.reshape((3, 4))        # 变为 3行4列 的矩阵
print(x)


# =================================================================
# 2. 算术运算与张量拼接 (Operations & Concatenation)
# =================================================================
x_val = torch.tensor([1.0, 2, 4, 5])
y_val = torch.tensor([2, 2, 2, 2])

# Element-wise (按元素运算)：形状必须相同（或符合广播规则）
print(x_val + y_val)         # 加减乘除均同理
print(x_val**y_val)          # 幂运算：[1^2, 2^2, 4^2, 5^2]
print(torch.exp(x_val))      # 指数运算 e^x

# 拼接逻辑：维度(dim)决定了增长的方向
# dim=0 (纵向堆叠): 增加行数； dim=1 (横向拼接): 增加列数
x_mat = torch.arange(12, dtype=torch.float32).reshape((3, 4))
y_mat = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

cat_dim0 = torch.cat((x_mat, y_mat), dim=0) # 结果形状 (6, 4)
cat_dim1 = torch.cat((x_mat, y_mat), dim=1) # 结果形状 (3, 8)


# =================================================================
# 3. 广播机制 (Broadcasting Mechanism)
# =================================================================
# 逻辑：当两个 Tensor 维度不符时，PyTorch 会自动复制元素来拉伸形状
# 只有当两个维度中，有一个为 1 时才能触发拉伸
a = torch.arange(3).reshape((3, 1)) # 3行1列
b = torch.arange(2).reshape((1, 2)) # 1行2列
# a 会复制列变为 (3,2)，b 会复制行变为 (3,2)，然后相加
print(a + b) 




# =================================================================
# 4. 内存管理与原地操作 (In-place Operations)
# =================================================================
# 内存陷阱：y = y + x 会重新分配内存，这在深度学习大数据处理时非常低效
before = id(y_mat)
y_mat = y_mat + x_mat
print(id(y_mat) == before)   # False：内存地址变了，创建了新对象

# 原地操作（In-place）优化：
z = torch.zeros_like(y_mat)
print('Initial id(z):', id(z))
z[:] = x_mat + y_mat         # 使用切片赋值，内存地址不变
print('Same id(z):', id(z))  # True：原地覆盖，节省内存开销


# =================================================================
# 5. 类型转换 (Conversion)
# =================================================================
# 与 NumPy 的互操作：在 CPU 上，两者共享底层存储内存
A = x_mat.numpy()            # Tensor -> NumPy
B = torch.tensor(A)          # NumPy -> Tensor

# 标量提取
a_scalar = torch.tensor([3.5])
print(a_scalar.item())       # .item() 提取为 Python 浮点数/整数
print(float(a_scalar))       # 强制类型转换同样有效

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
torch.Size([12])
<built-in method numel of Tensor object at 0x1251db340>
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([[ 2, 14,  4],
        [ 3,  6,  7],
        [ 1,  7,  8]])
tensor([3., 4., 6., 7.]) tensor([-1.,  0.,  2.,  3.]) tensor([ 2.,  4.,  8., 10.]) tensor([0.5000, 1.0000, 2.0000, 2.5000]) tensor([ 1.,  4., 16., 25.])
tensor([  2.7183,   7.3891,  54.5982, 148.4132])
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [ 2.,  1.,  4.,  3.],
        [ 1.,  2.,  3.,  4.],
        [ 4.,  3.,  2.,  1.]]) tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
        [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
        [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]])
tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])
<built-in method sum of Tensor object at 0x1251dbb60>
tensor([[0],
        

In [21]:
import os
import pandas as pd
import torch

# =================================================================
# 1. 模拟原始数据创建 (Data Creation)
# =================================================================
# 创建目录及 CSV 文件，模拟真实比赛或项目中的数据源
os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')

with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名：房间数, 巷道类型, 价格
    f.write('NA,Pave,127500\n')        # NA 表示缺失值
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

data = pd.read_csv(data_file)
print("--- 原始数据 ---")
print(data)


# =================================================================
# 2. 数据切片与缺失值处理 (Data Cleaning)
# =================================================================
# iloc 分离特征(inputs)与标签(outputs)
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]

# 逻辑：对于数值型特征（如 NumRooms），用该列的均值填充缺失值
# 注意：只有数值列能算 mean，文本列会被跳过
inputs = inputs.fillna(inputs.mean(numeric_only=True))
print("\n--- 均值填充后的 inputs ---")
print(inputs)


# =================================================================
# 3. 类别特征编码 (One-Hot Encoding)
# =================================================================
# 逻辑：计算机无法理解 "Pave" 或 "NA" 这种字符串。
# get_dummies 会将类别特征转换为多个二进制列（0 或 1）。
# dummy_na=True 表示将“缺失值”本身也作为一个合法的特征类别。
inputs = pd.get_dummies(inputs, dummy_na=True)
print("\n--- 独热编码后的 inputs ---")
print(inputs)


# =================================================================
# 4. 转换为 PyTorch 张量 (Conversion to Tensor)
# =================================================================
# 逻辑：深度学习模型要求输入必须是统一的数值矩阵。
# .values 将 pandas 对象转为 numpy 数组
# .astype('float32') 确保精度符合深度学习常用标准（32位浮点）
x = torch.tensor(inputs.values.astype('float32'))
y = torch.tensor(outputs.values.astype('float32'))

print("\n--- 最终生成的 PyTorch 张量 ---")
print(x, y)

--- 原始数据 ---
   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000

--- 均值填充后的 inputs ---
   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN

--- 独热编码后的 inputs ---
   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True

--- 最终生成的 PyTorch 张量 ---
tensor([[3., 1., 0.],
        [2., 0., 1.],
        [4., 0., 1.],
        [3., 0., 1.]]) tensor([127500., 106000., 178100., 140000.])


In [27]:
import torch

# =================================================================
# 1. 基础维度：标量 (Scalar) 与 向量 (Vector)
# =================================================================
a = torch.tensor(3.0)
b = torch.tensor(2.0)
print(a + b, a * b, a / b, a**b)  # 标量运算

x = torch.arange(4)              # 向量：一阶张量
print(x[3])                      # 通过索引访问元素
print(len(x))                    # 向量的长度


# =================================================================
# 2. 核心结构：矩阵 (Matrix) 与 张量 (Tensor)
# =================================================================
A = torch.arange(20).reshape(5, 4)
print(A)
print(A.T)                       # 矩阵转置 (Transpose)

# 验证对称矩阵 (Symmetric Matrix)
B = torch.tensor([[1, 2, 3], [2, 0, 4], [3, 4, 5]])
print(B == B.T)                  # 对称矩阵与其转置相等

# 多维张量（通常用于图像：通道 x 高 x 宽）
X = torch.arange(24).reshape(2, 3, 4)


# =================================================================
# 3. 降维与非降维求和 (Reduction & Non-Reduction)
# =================================================================
# 降维：sum() 会把张量压缩成标量
print(A.sum()) 

print('指定维度求和：')
# 指定维度求和：
# axis=0 沿行压缩（消除行维度，剩下列），axis=1 沿列压缩
A_sum_axis0 = A.sum(axis=0)      # 结果形状 (4,)
A_sum_axis1 = A.sum(axis=1)      # 结果形状 (5,)

#沿axis=i 求和， 就是把i这个维度去掉，比如，（2，3，4），axis=2，之后就变成（2，3）（axis从0开始）
#用keepdims的话就是保留被去掉的那个维度，他变成1，也就是变成（2，3，1）

# 非降维求和 (Keepdims)：保持原始维度，方便后续利用广播机制做归一化
sum_A = A.sum(axis=1, keepdims=True) 
print(A / sum_A)                 # 每一行被该行之和除（广播）


# =================================================================
# 4. 向量与矩阵乘法 (The "Meat" of Linear Algebra)
# =================================================================
x = torch.arange(4, dtype=torch.float32)
y = torch.ones(4, dtype=torch.float32)

# (1) 点积 (Dot Product)：相同位置相乘再求和
print(torch.dot(x, y))           # 结果为标量

# (2) 矩阵-向量积 (Matrix-Vector Product)
# A 是 (5,4), x 是 (4,) -> 结果是 (5,)
print(torch.mv(A.float(), x))

# (3) 矩阵-矩阵乘法 (Matrix-Matrix Multiplication)
# 本质：执行多次矩阵-向量积
B = torch.ones(4, 3)
print(torch.mm(A.float(), B))    # 结果形状 (5, 3)


# =================================================================
# 5. 范数 (Norms)：衡量张量的大小
# =================================================================
u = torch.tensor([3.0, -4.0])
# L2 范数：平方和再开根号（向量长度）
print(torch.norm(u))             

# L1 范数：绝对值之和
print(torch.abs(u).sum())

# 矩阵的 Frobenius 范数：矩阵元素平方和再开根号
print(torch.norm(torch.ones((4, 9))))

tensor(5.) tensor(6.) tensor(1.5000) tensor(9.)
tensor(3)
4
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])
tensor([[ 0,  4,  8, 12, 16],
        [ 1,  5,  9, 13, 17],
        [ 2,  6, 10, 14, 18],
        [ 3,  7, 11, 15, 19]])
tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])
tensor(190)
指定维度求和：
tensor([[0.0000, 0.1667, 0.3333, 0.5000],
        [0.1818, 0.2273, 0.2727, 0.3182],
        [0.2105, 0.2368, 0.2632, 0.2895],
        [0.2222, 0.2407, 0.2593, 0.2778],
        [0.2286, 0.2429, 0.2571, 0.2714]])
tensor(6.)
tensor([ 14.,  38.,  62.,  86., 110.])
tensor([[ 6.,  6.,  6.],
        [22., 22., 22.],
        [38., 38., 38.],
        [54., 54., 54.],
        [70., 70., 70.]])
tensor(5.)
tensor(7.)
tensor(6.)
